# 03 — Query planning, fusion, and reranking

## Scenario: rescue a vague incident question

A Northstar Cloud support engineer asks, *‘European checkout is slow after the release—what should I investigate?’* The knowledge base contains deployment notes, a regional latency incident, a generic health guide, and distractor documents. Build a retrieval system that broadens recall without losing the original intent, then spends richer ranking only where it can help.

The lab has no credentials or external services. It uses a transparent lexical baseline and a deterministic cross-encoder proxy so every score and decision can be inspected. Replace adapters with production models only after the experiment and release gates are in place.

## What you will learn

- Separate first-stage *candidate recall* from final *evidence ordering*.
- Keep the original question while producing a bounded, validated query plan.
- Fuse rankings with RRF when score scales are not comparable.
- Rerank a small candidate set with the original user intent.
- Measure recall@k, reciprocal rank, candidate depth, and operational budget.

```text
question -> original + bounded variants -> retrieve broadly
                                            |
                                            v
                                  deduplicate + RRF fusion
                                            |
                                            v
                                  rerank small candidate pool
                                            |
                                            v
                               cited evidence / abstention
```

**Invariant:** authorization filtering happens before every retrieval and reranking operation. This notebook focuses on ranking; combine it with the previous Metadata & Permissions lesson in a production pipeline.

## 1. Establish a reproducible corpus and relevance labels

Do not tune retrieval against a single demo query. Start with a small, versioned evaluation slice where the expected evidence IDs are known. The `regional-incident` and `deployment-842` documents are relevant to our scenario; the other documents are plausible distractors.

In [ ]:
from src.rag_core.lesson_loader import load_lesson_module; globals().update({name: value for name, value in vars(load_lesson_module('curriculum/intermediate/03-query-reranking/lab.py')).items() if not name.startswith('_')})

docs = [
    Document('regional-incident', 'European checkout timeout followed a dependency latency spike. Correlate regional logs and payment gateway latency.'),
    Document('deployment-842', 'At 08:42 checkout-api release 2026.8.4 changed payment retry configuration. Compare error rates before rollback.'),
    Document('support-sla', 'Enterprise customers receive a status update within 30 minutes of a confirmed incident.'),
    Document('health-guide', 'Use the health endpoint to check availability and dependency status.'),
    Document('catalog-release', 'Catalog search release improved synonym matching for product queries.'),
]
query = 'European checkout is slow after the release. What should support investigate?'
relevant_ids = {'regional-incident', 'deployment-842'}
[(doc.doc_id, doc.text) for doc in docs]

## 2. Plan queries without erasing the user's intent

A rewrite can recover corpus vocabulary—`slow` may map to `latency` or `timeout`—but it can also drop an entity, invent an assumption, or create extra cost. Keep the original query as the first variant. Limit the variant count, preserve IDs and explicit constraints, and log every generated variant.

A hosted LLM rewriter should produce a structured plan, not free-form prose: `variants`, `must_keep`, `intent`, and optionally `needs_clarification`. Validate those fields in application code and fall back to the original query on error.

In [ ]:
variants = rewrite_query(query)
for number, variant in enumerate(variants, start=1):
    print(f'{number}. {variant}')

assert variants[0] == query
assert len(variants) <= 4
assert any('latency' in variant or 'timeout' in variant for variant in variants)

## 3. Retrieve broadly, then fuse ranks

Each variant creates a ranking. Raw scores from different retrievers or query forms are commonly incomparable, so adding them can produce brittle behavior. Reciprocal Rank Fusion (RRF) combines rank positions instead. It rewards documents that occur near the top in multiple lists and leaves a provenance trail showing which variants contributed.

Candidate depth is a quality *and* operations control. Too shallow and the reranker cannot rescue missed evidence; too deep and cost and latency grow while irrelevant content competes for attention.

In [ ]:
candidates, first_stage_trace = retrieve_candidates(
    query, docs, top_k_per_variant=4, candidate_budget=5
)
for item in candidates:
    print({
        'id': item.document.doc_id,
        'rrf_score': round(item.score, 4),
        'best_variant_rank': item.first_stage_rank,
        'source_queries': len(item.source_queries),
    })

print(first_stage_trace)
print('candidate recall@5:', recall_at_k(candidates, relevant_ids, k=5))

## 4. Rerank only the bounded candidate pool

A cross-encoder scores a query and document together, which can model detailed relevance interactions that a bi-encoder or lexical score misses. It is intentionally a second stage because query-document pair scoring is expensive. The reusable module supplies a deterministic proxy so its behavior is inspectable; it is not a substitute for a trained reranker.

In production, choose a cross-encoder, a late-interaction model such as ColBERT, or a provider rerank API based on your language coverage, privacy boundaries, throughput, and evaluated quality. Pass the original question to the reranker, cap candidate count and document length, apply a timeout, and retain a safe fallback order.

In [ ]:
reranked = rerank(query, candidates, top_k=3)
for position, item in enumerate(reranked, start=1):
    print(f'{position}. {item.document.doc_id:18} rerank_score={item.score:.3f} first_stage_rank={item.first_stage_rank}')

print('final recall@3:', recall_at_k(reranked, relevant_ids, k=3))
print('final reciprocal rank:', reciprocal_rank(reranked, relevant_ids))
assert recall_at_k(reranked, relevant_ids, k=3) >= 0.5

## 5. Compare a bounded production pipeline

The `pipeline` helper records the query plan, candidate depth, and rerank count. A real trace should additionally include corpus/index revision, authorization-filter version, embedding/retriever/reranker versions, timings per stage, retries, cache status, and final context IDs. Do not store unauthorized document text in traces.

The important optimization target is not the lowest reranking cost. It is the smallest reliable candidate/rerank budget that meets a release threshold on the evaluation dataset and the difficult slices.

In [ ]:
final_results, trace = pipeline(
    query, docs, top_k_per_variant=4, candidate_budget=4, final_k=2
)
print(trace)
print('final IDs:', [item.document.doc_id for item in final_results])

# A deliberately simple latency model for design discussion; measure real services in production.
estimated_ms = 25 * len(trace.variants) + 45 * trace.candidate_count
print({'estimated_ms': estimated_ms, 'candidate_count': trace.candidate_count, 'rerank_count': trace.rerank_count})
assert trace.candidate_count <= 4
assert trace.rerank_count <= trace.candidate_count

## 6. Failure cases and release gates

| Failure | Symptom | Safer response |
| --- | --- | --- |
| Rewrite loses an error code | candidate recall falls for exact lookup | preserve identifiers; lexical fallback |
| More variants add noise | candidate set grows, MRR falls | lower rewrite budget; inspect variants |
| Reranker latency spikes | p95 breaches budget | reduce candidate depth or fall back to measured baseline |
| Score is treated as probability | unsafe confidence threshold | calibrate on held-out labels; use evidence policy |
| Unauthorized source is reranked | cross-tenant/context leak | filter before every candidate path |

### Exercises

1. Add an exact deployment identifier (`2026.8.4`) and show why retaining the original query matters.
2. Add a paraphrase-only query such as ‘the cart takes forever in the EU’; compare original-only and multi-query candidate recall.
3. Change `candidate_budget` from 2 to 5. Plot or record recall, MRR, and measured p95 latency before choosing a value.
4. Add a no-answer query. Define an abstention rule based on labeled relevance rather than always returning the top result.
5. Integrate the previous lesson’s authorization filter and prove a denied document never appears in candidates, reranked IDs, trace text, or cache.

### References

- [Sentence Transformers: Retrieve & Re-Rank](https://www.sbert.net/examples/sentence_transformer/applications/retrieve_rerank/README.html)
- [RRF paper](https://dl.acm.org/doi/10.1145/1571941.1572114)
- [Passage Re-ranking with BERT](https://arxiv.org/abs/1901.04085)
- [ColBERTv2](https://arxiv.org/abs/2112.01488)
- [Cohere Rerank overview](https://docs.cohere.com/docs/rerank-overview) (provider reference)